In [1]:
import random
from viz import draw_state
import numpy as np
from env import SimpleARGEnvironment
from rollout_worker_arg import RolloutWorker
from tb_gfn import TBGFlowNetGenerator
from utils import load_sequences

In [7]:
Ne = 10000
r_per_bp = 2e-8

dataset_path="/private/groups/corbettlab/pratik/git/ARG-Optimise_single_env/new_validation/fasta/sim_l1mb_0.fa"
sequences = load_sequences(dataset_path)
sequence_length = len(sequences[0])
num_blocks = 1000
rho = 4 * Ne * r_per_bp * num_blocks

In [8]:
rho

0.8

In [10]:
env = SimpleARGEnvironment(
        num_sequences=len(sequences),
        sequence_length=sequence_length,
        rho=rho,
        sequences=sequences,
        num_blocks=10000,
        fixed_edge_length=0.02,
        rng=random.Random(7),
    )

generator = TBGFlowNetGenerator(env, verbose=True)
rollout_worker = RolloutWorker(env, verbose=True)
batch_size = 2

Initializing log_Z from 2 prior rollout(s)...
step=01 action={'event_type': 'coal', 'active_lineage_i': 1, 'active_lineage_j': 5} log_prior=-3.4404 active=7 done=False
step=02 action={'event_type': 'coal', 'active_lineage_i': 0, 'active_lineage_j': 5} log_prior=-3.1697 active=6 done=False
step=03 action={'event_type': 'coal', 'active_lineage_i': 1, 'active_lineage_j': 3} log_prior=-2.8565 active=5 done=False
step=04 action={'event_type': 'coal', 'active_lineage_i': 0, 'active_lineage_j': 1} log_prior=-2.4849 active=4 done=False
step=05 action={'event_type': 'coal', 'active_lineage_i': 0, 'active_lineage_j': 1} log_prior=-2.0281 active=3 done=False
step=06 action={'event_type': 'recomb', 'active_lineage_i': 2, 'breakpoint': 1543} log_prior=-11.5616 active=4 done=False
step=07 action={'event_type': 'recomb', 'active_lineage_i': 1, 'breakpoint': 951} log_prior=-11.9511 active=5 done=False
step=08 action={'event_type': 'coal', 'active_lineage_i': 1, 'active_lineage_j': 3} log_prior=-2.1041

In [ ]:
ret, traj = rollout_worker.rollout(generator, episodes=batch_size)
states = ret["states"]
# draw_state(states[0])

In [ ]:
generator.accumulate_loss(ret)

In [ ]:
generator.loss

In [ ]:
last_info = generator.update_model()
# log_z = generator.compute_log_Z().detach().cpu().reshape(-1)[0].item()

In [ ]:
last_info

In [ ]:
draw_state(states[0])

In [ ]:
env.save_to_tree_sequence(states[0], "example.trees")

In [ ]:
act_seg = env.get_arg_sequence_segments(states[0])
[(b['child_node_id'], b['parent_node_ids']) for b in act_seg['recombination_events']]